In [3]:
#import libraries
import torch
import esm
import pandas as pd
import numpy as np 

In [ ]:
#load data
data = pd.read_csv("../../processed/combined_positives.csv")
data.head()

,mt_seq,wt_seq,label,data_source
0,ACDPHSGHFV,ARDPHSGHFV,1,CEDAR
1,AMLGTHTMEV,AMLSPHAMDV,1,CEDAR
2,EVDPIGHLY,EVVPISHLY,1,CEDAR
3,GVYDGEEHSV,GAYDGEEH,1,CEDAR
4,ITKKVADLVGF,KKVADLI,1,CEDAR


In [5]:
combined = data[["mt_seq", "wt_seq", "label"]]
combined.head()

,mt_seq,wt_seq,label
0,ACDPHSGHFV,ARDPHSGHFV,1
1,AMLGTHTMEV,AMLSPHAMDV,1
2,EVDPIGHLY,EVVPISHLY,1
3,GVYDGEEHSV,GAYDGEEH,1
4,ITKKVADLVGF,KKVADLI,1


In [6]:
#clean at dataset level

import re

def is_valid(seq):
    if not isinstance(seq, str):
        return False
    seq = seq.strip().upper()
    return bool(re.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]+", seq))

# Apply BOTH conditions together
df_clean = combined[
    combined["mt_seq"].apply(is_valid) &
    combined["wt_seq"].apply(is_valid)
].copy()

print("Before:", len(combined))
print("After:", len(df_clean))

Before: 11760
After: 11760


In [7]:
#convert into python lists
mt_seqs = df_clean["mt_seq"].tolist()
wt_seqs = df_clean["wt_seq"].tolist()
labels = df_clean["label"].values 

In [8]:
#set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [9]:
#load esm2 model
model_name = "esm2_t33_650M_UR50D"
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D() 
#move model to cpu
model = model.to(device)
#turn off traning mode
model.eval()
#convert seq to tokens
batch_converter = alphabet.get_batch_converter()

In [10]:
#mean pooling function
def mean_pool(token_representations, seq_len): 
    #converts per aa per token to single vector per seq
    #ignores special tokens
    return token_representations[1:seq_len+1].mean(0).cpu().numpy() 

#embedding function
def embed_seq(sequences, batch_size=14): 
    all_embeddings = []
    #loop over btatches
    for i in range(0, len(sequences), batch_size): 
        batch = sequences[i:i+batch_size]
        #ESM format
        data = [(str(i), seq) for i, seq in enumerate(batch)]
        #convert to tokens
        _, _, tokens = batch_converter(data)
        tokens = tokens.to(device)
        #disable gradients
        with torch.no_grad():
            results = model(tokens, repr_layers=[33]) 
        #extract embeddings from layer 33
        token_reps = results["representations"][33]
        #convert each seq in batch
        for j, seq in enumerate(batch): 
            emb = mean_pool(token_reps[j], len(seq))
            all_embeddings.append(emb)
    return np.array(all_embeddings)


In [11]:
#generate embeddings
mt_embeddings = embed_seq(mt_seqs)
wt_embeddings = embed_seq(wt_seqs)

In [ ]:
#save output
np.savez("../../output/embeddings/esm2_embeddings_combined.npz", 
mt_mean = mt_embeddings, wt_mean = wt_embeddings, 
labels = labels)